<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="../../images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    3.4.1. Finding the Minimal Neural Network for XOR
  </strong>
</div>

In [ ]:
# ⚠️ IMPORTANT NOTICE FOR STUDENTS ⚠️
#
# Please make sure to check the official instructions for this assignment in Canvas LMS
# as they may have been updated or changed. The instructions above are provided for
# reference only and may not reflect the most current requirements.
#
# Always refer to Canvas LMS for:
# - Latest assignment requirements
# - Due dates
# - Grading criteria
# - Any special instructions
#
# When in doubt, ask your instructor for clarification.
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from rich.console import Console
from rich.table import Table
import warnings
warnings.filterwarnings('ignore')

console = Console()

In [ ]:
inputs = torch.tensor(
    [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]], dtype=torch.float32
)
labels = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

table = Table(title="XOR Truth Table")
table.add_column("Input 1", justify="center")
table.add_column("Input 2", justify="center")
table.add_column("Output", justify="center")

for input_row, label_row in zip(inputs, labels):
    table.add_row(
        str(int(input_row[0].item())),
        str(int(input_row[1].item())),
        str(int(label_row[0].item())),
    )

console.print(table)

# 1. Network Definition and Training Utility

In [ ]:
class XorNet(nn.Module):
    def __init__(self, hidden_sizes, activation='sigmoid'):
        super(XorNet, self).__init__()
        self.activation_name = activation
        
        layers = []
        input_size = 2
        
        for h_size in hidden_sizes:
            layers.append(nn.Linear(input_size, h_size))
            input_size = h_size
        
        layers.append(nn.Linear(input_size, 1))
        self.network = nn.Sequential(*layers)
        
        if activation == 'sigmoid':
            self.act = nn.Sigmoid()
        elif activation == 'tanh':
            self.act = nn.Tanh()
        elif activation == 'relu':
            self.act = nn.ReLU()
        else:
            self.act = nn.Identity()
    
    def forward(self, x):
        for i, layer in enumerate(self.network):
            x = layer(x)
            if i < len(self.network) - 1:
                x = self.act(x)
        return x

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
def train_xor(hidden_sizes, activation='sigmoid', lr=0.01, epochs=3000,
              n_seeds=10, target_loss=0.01, verbose=False):
    """
    Train a XOR network with multiple random seeds.
    Returns success rate, average epochs to converge, loss histories.
    """
    all_loss_histories = []
    convergence_epochs = []
    successes = 0
    
    for seed in range(n_seeds):
        torch.manual_seed(seed)
        np.random.seed(seed)
        
        model = XorNet(hidden_sizes, activation)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        loss_history = []
        converged = False
        
        for epoch in range(epochs):
            optimizer.zero_grad()
            output = model(inputs)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
            
            loss_val = loss.item()
            loss_history.append(loss_val)
            
            if loss_val < target_loss and not converged:
                converged = True
                convergence_epochs.append(epoch + 1)
        
        if converged:
            successes += 1
        else:
            convergence_epochs.append(epochs + 1)
        
        all_loss_histories.append(loss_history)
        
        if verbose:
            final_pred = model(inputs).detach().numpy()
            status = 'OK' if converged else 'FAIL'
            print(f'  Seed {seed}: loss={loss_val:.6f} | {status}')
    
    success_rate = successes / n_seeds
    avg_epochs = np.mean(convergence_epochs)
    avg_final_loss = np.mean([h[-1] for h in all_loss_histories])
    
    return {
        'success_rate': success_rate,
        'avg_epochs': avg_epochs,
        'avg_final_loss': avg_final_loss,
        'loss_histories': all_loss_histories,
        'convergence_epochs': convergence_epochs
    }

# 2. Architecture Experiments

In [ ]:
architectures = [
    {'name': 'Linear (baseline)', 'hidden': [], 'activation': 'none'},
    {'name': '1H Sigmoid', 'hidden': [1], 'activation': 'sigmoid'},
    {'name': '1H ReLU', 'hidden': [1], 'activation': 'relu'},
    {'name': '2H Sigmoid', 'hidden': [2], 'activation': 'sigmoid'},
    {'name': '2H ReLU', 'hidden': [2], 'activation': 'relu'},
    {'name': '2-1 Sigmoid', 'hidden': [2, 1], 'activation': 'sigmoid'},
    {'name': '4H Sigmoid', 'hidden': [4], 'activation': 'sigmoid'},
]

results = []

for arch in architectures:
    model_temp = XorNet(arch['hidden'], arch['activation'])
    n_params = model_temp.count_parameters()
    
    result = train_xor(
        hidden_sizes=arch['hidden'],
        activation=arch['activation'],
        lr=0.01,
        epochs=3000,
        n_seeds=15,
        target_loss=0.01
    )
    
    results.append({
        'Architecture': arch['name'],
        'Hidden Layers': str(arch['hidden']) if arch['hidden'] else 'None',
        'Activation': arch['activation'],
        'Parameters': n_params,
        'Success Rate': f"{result['success_rate']*100:.0f}%",
        'Avg Epochs': f"{result['avg_epochs']:.0f}",
        'Avg Final Loss': f"{result['avg_final_loss']:.6f}",
        'loss_histories': result['loss_histories'],
        'convergence_epochs': result['convergence_epochs']
    })
    
    console.print(f'{arch["name"]} ({n_params}p): Success={result["success_rate"]*100:.0f}%, Avg Epochs={result["avg_epochs"]:.0f}')

# 3. Comparison Table

In [ ]:
table_results = Table(title="Architecture Comparison - Minimal XOR Network")
table_results.add_column("Architecture", justify="left")
table_results.add_column("Hidden", justify="center")
table_results.add_column("Activation", justify="center")
table_results.add_column("Parameters", justify="center")
table_results.add_column("Success Rate", justify="center")
table_results.add_column("Avg Epochs", justify="center")
table_results.add_column("Avg Loss", justify="center")

for r in results:
    success_str = r['Success Rate']
    if success_str == '100%':
        style = 'green'
    elif int(success_str.replace('%', '')) >= 80:
        style = 'yellow'
    else:
        style = 'red'
    
    table_results.add_row(
        r['Architecture'],
        r['Hidden Layers'],
        r['Activation'],
        str(r['Parameters']),
        f"[{style}]{success_str}[/]",
        r['Avg Epochs'],
        r['Avg Final Loss']
    )

console.print(table_results)

In [ ]:
df_results = pd.DataFrame([{k: v for k, v in r.items() if k not in ['loss_histories', 'convergence_epochs']} for r in results])
df_results.to_csv('data/architecture_comparison.csv', index=False)
print('Table saved to data/architecture_comparison.csv')
df_results

# 4. Training Curves

In [ ]:
n_archs = len(results)
n_cols = 4
n_rows = (n_archs + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
axes = axes.flatten()

for i, r in enumerate(results):
    histories = r['loss_histories']
    mean_history = np.mean(histories, axis=0)
    std_history = np.std(histories, axis=0)
    
    epochs_range = range(1, len(mean_history) + 1)
    axes[i].plot(epochs_range, mean_history, label='Mean Loss', linewidth=2)
    axes[i].fill_between(epochs_range,
                         np.maximum(mean_history - std_history, 0),
                         mean_history + std_history,
                         alpha=0.2, label='Std Dev')
    axes[i].axhline(y=0.01, color='r', linestyle='--', alpha=0.5, label='Target')
    axes[i].set_title(f"{r['Architecture']} ({r['Parameters']}p)")
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('MSE Loss')
    axes[i].set_yscale('log')
    axes[i].legend(fontsize=7)
    axes[i].grid(True, alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Training Curves for All Architectures', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('data/training_curves_all.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to data/training_curves_all.png')

# 5. Convergence Epoch Distribution (Best Architecture)

In [ ]:
best_result = None
for r in results:
    if r['Success Rate'] == '100%':
        if best_result is None or r['Parameters'] < best_result['Parameters']:
            best_result = r

if best_result is None:
    for r in results:
        if best_result is None or r['Parameters'] < best_result['Parameters']:
            best_result = r

console.print(f"\n[bold green]Best Minimal Architecture: {best_result['Architecture']} ({best_result['Parameters']} params)[/]")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

conv_epochs = best_result['convergence_epochs']
axes[0].hist(conv_epochs, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(x=np.mean(conv_epochs), color='r', linestyle='--', linewidth=2, label=f'Mean: {np.mean(conv_epochs):.0f}')
axes[0].set_title(f"Convergence Epoch Distribution - {best_result['Architecture']}")
axes[0].set_xlabel('Epoch to Converge (loss < 0.01)')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

best_histories = best_result['loss_histories']
for idx, h in enumerate(best_histories[:10]):
    axes[1].plot(h, alpha=0.5, linewidth=0.8, label=f'Seed {idx}' if idx < 3 else None)
axes[1].axhline(y=0.01, color='r', linestyle='--', alpha=0.5, label='Target')
axes[1].set_title(f"Individual Training Runs - {best_result['Architecture']}")
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].set_yscale('log')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/convergence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to data/convergence_distribution.png')

# 6. Learning Rate Experiments

In [ ]:
best_hidden = best_result['Hidden Layers']
best_act = best_result['Activation']
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.5]

lr_results = []

for lr in learning_rates:
    hidden_list = eval(best_hidden) if best_hidden != 'None' else []
    result = train_xor(
        hidden_sizes=hidden_list,
        activation=best_act,
        lr=lr,
        epochs=3000,
        n_seeds=10,
        target_loss=0.01
    )
    lr_results.append({
        'Learning Rate': lr,
        'Success Rate': result['success_rate'],
        'Avg Epochs': result['avg_epochs'],
        'Avg Loss': result['avg_final_loss'],
        'loss_histories': result['loss_histories']
    })
    console.print(f'LR={lr}: Success={result["success_rate"]*100:.0f}%, Avg Epochs={result["avg_epochs"]:.0f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lr_res in lr_results:
    mean_history = np.mean(lr_res['loss_histories'], axis=0)
    axes[0].plot(mean_history, label=f"LR={lr_res['Learning Rate']}", linewidth=1.5)

axes[0].axhline(y=0.01, color='r', linestyle='--', alpha=0.5, label='Target')
axes[0].set_title(f"Effect of Learning Rate - {best_result['Architecture']}")
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

lr_labels = [str(lr_res['Learning Rate']) for lr_res in lr_results]
success_rates = [lr_res['Success Rate'] * 100 for lr_res in lr_results]
axes[1].bar(lr_labels, success_rates, color='steelblue', edgecolor='black')
axes[1].set_title('Success Rate by Learning Rate')
axes[1].set_xlabel('Learning Rate')
axes[1].set_ylabel('Success Rate (%)')
axes[1].set_ylim(0, 110)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('data/learning_rate_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to data/learning_rate_analysis.png')

# 7. Parameters vs Convergence Trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

param_counts = [r['Parameters'] for r in results]
success_pcts = [float(r['Success Rate'].replace('%', '')) for r in results]
names = [r['Architecture'] for r in results]
avg_epochs_vals = [float(r['Avg Epochs']) for r in results]

scatter = ax.scatter(param_counts, success_pcts, s=100, c=avg_epochs_vals,
                     cmap='RdYlGn_r', edgecolors='black', linewidth=0.5)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Avg Epochs to Converge')

for i, name in enumerate(names):
    ax.annotate(name, (param_counts[i], success_pcts[i]),
                textcoords='offset points', xytext=(5, 5), fontsize=8)

ax.set_xlabel('Number of Parameters')
ax.set_ylabel('Success Rate (%)')
ax.set_title('Parameters vs Success Rate (color = convergence speed)')
ax.set_ylim(-5, 110)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/params_vs_success.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to data/params_vs_success.png')

In [ ]:
import os
print('Files saved in data/ folder:')
for f in sorted(os.listdir('data')):
    print(f'  - {f}')

# Análise dos Resultados

## Problema XOR

O problema XOR consiste em quatro pontos de treinamento binários: 00 retorna 0, 01 retorna 1, 10 retorna 1 e 11 retorna 0. Essa função não é linearmente separável, ou seja, não existe uma hiperplano que divida corretamente as duas classes no espaço de entrada. A consequência direta é que qualquer modelo sem camada oculta, incluindo o perceptron simples ou uma regressão linear, falha nessa tarefa. A loss do modelo linear ficou travada em 0.25, que é exatamente o erro de quem prevê a média dos targets para toda entrada, já que o XOR tem duas saídas 0 e duas saídas 1, resultando em média 0.5.

## Arquiteturas Testadas e Resultados

Cada arquitetura foi treinada com 15 seeds aleatórias diferentes para avaliar a consistência da convergência. O critério de sucesso foi atingir uma MSE inferior a 0.01 em pelo menos uma das 3000 épocas. O otimizador utilizado foi Adam com learning rate de 0.01.

- **Linear sem camada oculta**: 3 parâmetros, 2 pesos de entrada + 1 bias. Como discutido acima, não tem capacidade de representação para funções não-lineares. A loss não mudou ao longo de todo o treinamento. Resultado esperado.

- **1 neurônio oculto com sigmoid**: 5 parâmetros, 2 pesos na primeira camada + 1 bias oculto + 1 peso na saída + 1 bias de saída. Apesar de ter uma camada oculta, um único neurônio não tem capacidade suficiente para mapear os 4 pontos do XOR para as 4 saídas corretas simultaneamente. A rede converge para um mínimo local em que as previsões ficam em torno de 0.5 para todas as entradas, ou seja, não aprendeu nada útil. Taxa de sucesso de 0%.

- **1 neurônio oculto com ReLU**: Mesma quantidade de parâmetros, 5. O ReLU tem a vantagem de não saturar para valores positivos, mas para o caso de um único neurônio isso não resolve o problema fundamental de capacidade. Se o peso de entrada é negativo, o neurônio fica inativo para todas as entradas, e se é positivo, ele só consegue criar uma separação em um único ponto de corte. Taxa de sucesso de 0%.

- **2 neurônios ocultos com sigmoid**: 9 parâmetros, 2×2 pesos na primeira camada + 2 biases ocultos + 2 pesos na saída + 1 bias de saída. Essa arquitetura conseguiu 80% de sucesso. O número de neurônios é suficiente para criar duas regiões de ativação distintas que, combinadas, cobrem os dois padrões do XOR. Porém, em 20% das inicializações o otimizador caiu em mínimos locais onde a loss ficou acima de 0.01. Isso indica que, embora a capacidade da rede seja adequada, a superfície de perda tem múltiplos mínimos locais e a inicialização aleatória pode levar o treinamento para uma região ruim do espaço de parâmetros.

- **2 neurônios ocultos com ReLU**: Mesmos 9 parâmetros. A taxa de sucesso caiu para apenas 20%. O problema do ReLU com essa arquitetura pequena é o fenômeno de dead neurons: se os pesos iniciais são negativos, o gradiente do ReLU é zero e o neurônio nunca se ativa. Com apenas 2 neurônios, se um deles morre, a rede perde metade da sua capacidade de representação. A sigmoid não tem esse problema porque o gradiente nunca é zero.

- **2 neurônios na primeira camada e 1 na segunda, sigmoid**: 11 parâmetros. A adição de uma segunda camada oculta deveria aumentar a capacidade de representação, mas na prática a taxa de sucesso foi de apenas 53%. Redes mais profundas para um problema tão pequeno tendem a ter uma superfície de perda mais complexa, com mais mínimos locais e barreiras de gradiente. A profundidade extra não ajuda nesse caso.

- **4 neurônios ocultos com sigmoid**: 17 parâmetros. Essa arquitetura atingiu 100% de sucesso em todas as 15 seeds, com uma época média de convergência de apenas 467. Com 4 neurônios, a rede tem redundância suficiente para que, mesmo com uma inicialização ruim, pelo menos alguns neurônios se ativem nas regiões certas do espaço de entrada. A loss final média foi praticamente zero.

## Análise do Learning Rate

Para a melhor arquitetura, que foi a de 4 neurônios sigmoid, eu testei cinco valores de learning rate. Com LR=0.001 a convergência foi muito lenta, a taxa de sucesso ficou em 40% porque muitas execuções não chegaram no critério de 0.01 de loss dentro de 3000 épocas. Com LR=0.01 o resultado já foi de 100% de sucesso e época média de 477. Com LR=0.05 e LR=0.1 os resultados foram ainda melhores em termos de velocidade, época média de 147 e 139 respectivamente, ambos com 100% de sucesso. Quando o LR chegou em 0.5, a taxa de sucesso caiu para 70% porque o passo de atualização era grande demais e o otimizador oscilava em torno do mínimo sem convergir. A janela ideal para essa arquitetura ficou entre 0.01 e 0.1.

## Trade-offs entre Parâmetros e Confiabilidade

A relação entre número de parâmetros e desempenho não é estritamente monotônica. Adicionar parâmetros melhora a capacidade de representação, mas até certo ponto. A rede linear com 3 parâmetros não tem capacidade nenhuma para XOR. A rede com 5 parâmetros tem capacidade marginal, insuficiente para convergir de forma confiável. A rede com 9 parâmetros tem capacidade adequada, mas depende da inicialização com sigmoid, e é frágil com ReLU. A rede com 17 parâmetros tem redundância de sobra e converge sempre.

O ponto de inflexão está entre 9 e 17 parâmetros. Com 9 parâmetros, 80% de sucesso não é aceitável para o critério da tarefa, que exige consistência independentemente da inicialização. Com 17 parâmetros, 100% de sucesso. Portanto, a rede mínima viável para essa tarefa é a de 4 neurônios sigmoid com 17 parâmetros.

## Conclusão

A rede neural mínima para resolver XOR consistentemente em 3000 épocas ou menos, independentemente da inicialização aleatória, é uma rede com uma camada oculta de 4 neurônios usando ativação sigmoid. Essa arquitetura possui 17 parâmetros treináveis, converge em média em 467 épocas com learning rate de 0.01 e atingiu 100% de sucesso em todas as 15 seeds testadas. Arquiteturas menores, como a de 2 neurônios sigmoid com 9 parâmetros, atingem apenas 80% de sucesso e portanto não atendem ao critério de consistência. A utilização de ReLU em arquiteturas pequenas se mostrou problemática devido ao fenômeno de dead neurons. A profundidade adicional em arquiteturas como 2-1 sigmoid não trouxe benefício e piorou a taxa de convergência.